# LLM Evaluation on Kaggle

Notebook này dùng riêng cho Kaggle Notebook để benchmark các LLM trên comparative quintuple extraction.

Trước khi chạy:
1. Add project folder dưới dạng Kaggle Input Dataset hoặc clone từ GitHub.
2. Thêm `OPENROUTER_API_KEY` trong `Settings -> Secrets`.
3. Chỉnh `PROJECT_INPUT_DIR`, `DATASETS`, `SPLIT`, `MODELS`, `PROMPT_STRATEGY` ở cell cấu hình.

## Nên dùng openrouter hay hf-local?

- Dùng `openrouter` khi:
  - Bạn muốn kết quả ổn định và không bị giới hạn bởi VRAM notebook.
  - Bạn cần chạy nhiều model API trong thời gian ngắn.
  - Bạn chấp nhận chi phí theo token.
- Dùng `hf-local` khi:
  - Bạn muốn tối ưu chi phí API và chấp nhận chi phí compute local.
  - Bạn chỉ benchmark model open-source từ Hugging Face.
  - Bạn muốn bật quantization (`HF_LOAD_IN_4BIT`) để tiết kiệm VRAM.

## Gợi ý VRAM cho hf-local (tham khảo)

- 3B model:
  - FP16/BF16: >= 8 GB VRAM
  - 4-bit: >= 4-6 GB VRAM
- 7B-8B model:
  - FP16/BF16: >= 16 GB VRAM
  - 4-bit: >= 8-12 GB VRAM
- 13B model:
  - FP16/BF16: >= 24 GB VRAM
  - 4-bit: >= 12-16 GB VRAM
- 30B+ model:
  - Thường cần multi-GPU hoặc không phù hợp với Kaggle GPU phổ biến.

Lưu ý:
- Kaggle thường phù hợp với `hf-local` + 4-bit cho model <= 7B/8B.
- Nếu báo OOM, ưu tiên đổi model nhỏ hơn trước.
- Hãy chạy `LIMIT > 0` để smoke test trước khi chạy full test set.

## Model gợi ý từ Hugging Face (EN + VI)

Shortlist cân bằng chất lượng/chi phí:
1. `Qwen/Qwen2.5-3B-Instruct`
2. `Qwen/Qwen2.5-7B-Instruct`
3. `mistralai/Mistral-7B-Instruct-v0.3`

Nếu có GPU tốt hơn, có thể mở rộng:
- `Qwen/Qwen2.5-14B-Instruct`
- `meta-llama/Llama-3.1-8B-Instruct`

## Benchmark matrix 3 x 3 đề xuất (so sánh E-T5-MACRO-F1)

- Models:
  - `Qwen/Qwen2.5-3B-Instruct`
  - `Qwen/Qwen2.5-7B-Instruct`
  - `mistralai/Mistral-7B-Instruct-v0.3`
- Strategies:
  - `zero-shot`
  - `few-shot`
  - `cot`

Tổng cộng 9 run:
1. 3B x zero-shot
2. 3B x few-shot
3. 3B x cot
4. 7B x zero-shot
5. 7B x few-shot
6. 7B x cot
7. Mistral-7B x zero-shot
8. Mistral-7B x few-shot
9. Mistral-7B x cot

In [ ]:
# Cell 1 · Kaggle paths
import os

assert os.path.exists('/kaggle/working'), 'Notebook này chỉ dành cho Kaggle.'

PROJECT_INPUT_DIR = '/kaggle/input/msc-project'
WORK_DIR = '/kaggle/working/msc-project'
LLMEVAL_DIR = os.path.join(WORK_DIR, 'llm_eval')
DATASETS_ROOT = os.path.join(WORK_DIR, 'datasets')
OUTPUT_DIR = os.path.join(LLMEVAL_DIR, 'results')
CACHE_DIR = os.path.join(LLMEVAL_DIR, 'cache')

print('Input project dir:', PROJECT_INPUT_DIR)
print('Working project dir:', WORK_DIR)

In [ ]:
# Cell 2 · Prepare project in /kaggle/working
GITHUB_REPO = ''  # tùy chọn: clone nếu không dùng Kaggle Input

import os
import shutil
import subprocess

if not os.path.exists(LLMEVAL_DIR):
    if os.path.exists(PROJECT_INPUT_DIR):
        shutil.copytree(PROJECT_INPUT_DIR, WORK_DIR, dirs_exist_ok=True)
        print('Copied project from Kaggle Input Dataset.')
    elif GITHUB_REPO:
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, WORK_DIR], check=True)
        print('Cloned project from GitHub.')
    else:
        raise FileNotFoundError('Không tìm thấy project trong Kaggle Input và GITHUB_REPO đang để trống.')
else:
    print(f'Project already exists at {WORK_DIR}')

In [ ]:
# Cell 3 · Install dependencies
import os
import subprocess
import sys

reqs = os.path.join(LLMEVAL_DIR, 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', reqs, '-q'], check=True)
print('Dependencies installed.')

In [ ]:
# Cell 4 · Load OpenRouter API key from Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient

os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY not found in Kaggle Secrets.'
print('API key loaded.')

## Khối A: HF-Local Workflow (Sử dụng mô hình cục bộ - tiết kiệm chi phí)
Chạy các ô dưới đây nếu bạn muốn sử dụng HuggingFace models cục bộ. Kaggle cung cấp GPU với ~30GB VRAM, đủ cho các mô hình 7B-14B.

In [ ]:
# [A.1] HF-Local Configuration
print("=" * 100)
print("KHỐI A: HF-LOCAL WORKFLOW")
print("=" * 100)

DATASETS = 'camera-coqe,vcom-data'
SPLIT = 'test'
PROMPT_STRATEGY = 'few-shot'  # zero-shot | few-shot | cot

# Provider settings for HF-Local
PROVIDER = 'hf-local'
HF_DTYPE = 'auto'             # auto | float16 | bfloat16
HF_LOAD_IN_4BIT = False       # Set to True if VRAM is limited

# HF-Local Model recommendations:
# - Qwen/Qwen2.5-3B-Instruct (3B, lightweight, ~7-8GB GPU)
# - Qwen/Qwen2.5-7B-Instruct (7B, balanced, ~15-16GB GPU)
# - Qwen/Qwen2.5-14B-Instruct (14B, powerful, ~28-30GB GPU)
# - mistralai/Mistral-7B-Instruct-v0.2 (7B, fast, ~15-16GB GPU)
# Kaggle provides ~30GB GPU (usually P100)

MODELS = [
    'Qwen/Qwen2.5-3B-Instruct',
    'Qwen/Qwen2.5-7B-Instruct',
]

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 256
SLEEP_SECONDS = 0.3
LIMIT = 0

print('Configuration ready for HF-Local.')
print(f'Provider: {PROVIDER}')
print(f'Models: {MODELS}')
print(f'Strategy: {PROMPT_STRATEGY}')
print(f'HF_DTYPE: {HF_DTYPE}, HF_LOAD_IN_4BIT: {HF_LOAD_IN_4BIT}')

In [ ]:
# [A.2] HF-Local: Single-sentence smoke test
import os
import sys

SMOKE_SENTENCE = 'Bên cạnh đó, iPhone 14 được nâng cấp bộ nhớ lên đến 6GB RAM cao hơn iPhone 13 đến 2GB RAM, cho khả năng đa nhiệm tốt hơn.'
SMOKE_DATASET = 'vcom-data'      # vcom-data | camera-coqe
SMOKE_LANGUAGE = 'vi'            # vi | en | auto
SMOKE_STRATEGY = PROMPT_STRATEGY

sys.path.insert(0, LLMEVAL_DIR)
from prompts import build_messages
from client import HuggingFaceLocalClient

messages = build_messages(
    sentence=SMOKE_SENTENCE,
    language=SMOKE_LANGUAGE,
    dataset=SMOKE_DATASET,
    strategy=SMOKE_STRATEGY,
)

print(f"\n[HF-Local Smoke Test] Testing with model: {MODELS[0]}")
smoke_client = HuggingFaceLocalClient(
    model=MODELS[0],
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    dtype=HF_DTYPE,
    load_in_4bit=HF_LOAD_IN_4BIT,
)

smoke_output = smoke_client.generate(messages)

print('Provider :', PROVIDER)
print('Model    :', MODELS[0])
print('Strategy :', SMOKE_STRATEGY)
print('Sentence :', SMOKE_SENTENCE)
print('-' * 100)
print('Output:')
print(smoke_output)

In [ ]:
# [A.3] HF-Local: Full evaluation
import os
import subprocess
import sys

print("\n[HF-Local] Running full evaluation...")
cmd = [
    sys.executable, os.path.join(LLMEVAL_DIR, 'run_eval.py'),
    '--datasets', DATASETS,
    '--split', SPLIT,
    '--models', *MODELS,
    '--provider', PROVIDER,
    '--prompt-strategy', PROMPT_STRATEGY,
    '--temperature', str(TEMPERATURE),
    '--max-output-tokens', str(MAX_OUTPUT_TOKENS),
    '--sleep-seconds', str(SLEEP_SECONDS),
    '--datasets-root', DATASETS_ROOT,
    '--output-dir', OUTPUT_DIR,
    '--cache-dir', CACHE_DIR,
    '--hf-dtype', HF_DTYPE,
]

if HF_LOAD_IN_4BIT:
    cmd += ['--hf-load-in-4bit']

if LIMIT > 0:
    cmd += ['--limit', str(LIMIT)]

print('Running command:')
print(' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=False)
print('Exit code:', result.returncode)

## Khối B: OpenRouter Workflow (Sử dụng API - các mô hình lớn)
Chạy các ô dưới đây nếu bạn muốn sử dụng OpenRouter API để gọi các mô hình mạnh mẽ (GPT-4, Claude, Gemini, v.v.).

In [ ]:
# [B.1] OpenRouter Configuration
print("=" * 100)
print("KHỐI B: OPENROUTER WORKFLOW")
print("=" * 100)

DATASETS = 'camera-coqe,vcom-data'
SPLIT = 'test'
PROMPT_STRATEGY = 'few-shot'  # zero-shot | few-shot | cot

# Provider settings for OpenRouter
PROVIDER = 'openrouter'
# OpenRouter API key should already be in environment

# OpenRouter model options:
# - openai/gpt-4o-mini (fast, cost-effective)
# - anthropic/claude-3.5-haiku (multimodal, reliable)
# - google/gemini-2.0-flash-001 (fast, powerful)
# - deepseek/deepseek-chat (efficient)
# - qwen/qwen-2.5-72b-instruct (very powerful)
# - meta-llama/llama-3.3-70b-instruct (open-source alternative)

MODELS = [
    'openai/gpt-4o-mini',
    'anthropic/claude-3.5-haiku',
    'google/gemini-2.0-flash-001',
]

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 256
SLEEP_SECONDS = 0.3  # Respect rate limits
LIMIT = 0

print('Configuration ready for OpenRouter.')
print(f'Provider: {PROVIDER}')
print(f'Models: {MODELS}')
print(f'Strategy: {PROMPT_STRATEGY}')
print('WARNING: OpenRouter requires OPENROUTER_API_KEY environment variable to be set!')

In [ ]:
# [B.2] OpenRouter: Single-sentence smoke test
import os
import sys

SMOKE_SENTENCE = 'Bên cạnh đó, iPhone 14 được nâng cấp bộ nhớ lên đến 6GB RAM cao hơn iPhone 13 đến 2GB RAM, cho khả năng đa nhiệm tốt hơn.'
SMOKE_DATASET = 'vcom-data'      # vcom-data | camera-coqe
SMOKE_LANGUAGE = 'vi'            # vi | en | auto
SMOKE_STRATEGY = PROMPT_STRATEGY

sys.path.insert(0, LLMEVAL_DIR)
from prompts import build_messages
from client import OpenAICompatibleClient

messages = build_messages(
    sentence=SMOKE_SENTENCE,
    language=SMOKE_LANGUAGE,
    dataset=SMOKE_DATASET,
    strategy=SMOKE_STRATEGY,
)

print(f"\n[OpenRouter Smoke Test] Testing with model: {MODELS[0]}")
smoke_client = OpenAICompatibleClient(
    model=MODELS[0],
    base_url='https://openrouter.ai/api/v1',
    api_key_env='OPENROUTER_API_KEY',
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

smoke_output = smoke_client.generate(messages)

print('Provider :', PROVIDER)
print('Model    :', MODELS[0])
print('Strategy :', SMOKE_STRATEGY)
print('Sentence :', SMOKE_SENTENCE)
print('-' * 100)
print('Output:')
print(smoke_output)

In [ ]:
# [B.3] OpenRouter: Full evaluation
import os
import subprocess
import sys

print("\n[OpenRouter] Running full evaluation...")
cmd = [
    sys.executable, os.path.join(LLMEVAL_DIR, 'run_eval.py'),
    '--datasets', DATASETS,
    '--split', SPLIT,
    '--models', *MODELS,
    '--provider', PROVIDER,
    '--prompt-strategy', PROMPT_STRATEGY,
    '--temperature', str(TEMPERATURE),
    '--max-output-tokens', str(MAX_OUTPUT_TOKENS),
    '--sleep-seconds', str(SLEEP_SECONDS),
    '--datasets-root', DATASETS_ROOT,
    '--output-dir', OUTPUT_DIR,
    '--cache-dir', CACHE_DIR,
    '--base-url', 'https://openrouter.ai/api/v1',
    '--api-key-env', 'OPENROUTER_API_KEY',
]

if LIMIT > 0:
    cmd += ['--limit', str(LIMIT)]

print('Running command:')
print(' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=False)
print('Exit code:', result.returncode)

## Summary & Output
Xem kết quả đánh giá từ Khối A hoặc Khối B.

In [ ]:
# Cell 7 · Show summary
import json
import pathlib

summary_file = pathlib.Path(OUTPUT_DIR) / f'summary__{SPLIT}.json'

if summary_file.exists():
    with open(summary_file, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    try:
        import pandas as pd
        df = pd.DataFrame([
            {
                'dataset': r['dataset'],
                'model': r['model'],
                'E-T5-MACRO-F1': round(r.get('E-T5-MACRO-F1', 0), 4),
                'E-T4-F1': round(r.get('E-T4-F1', 0), 4),
                'E-CEE-MICRO-F1': round(r.get('E-CEE-MICRO-F1', 0), 4),
            }
            for r in rows
        ]).sort_values(['dataset', 'E-T5-MACRO-F1'], ascending=[True, False])
        print(df.to_string(index=False))
    except ImportError:
        print(rows)
else:
    print('Summary file not found.')

In [ ]:
# Cell 8 · Results location
import pathlib

results_path = pathlib.Path(OUTPUT_DIR)
print('Results saved at:', results_path)
print('Use the right-side Output/Data panel in Kaggle to download files if needed.')